In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from state import State

In [2]:
from langchain_mcp_adapters.tools import load_mcp_tools

In [3]:
mcp_client = MultiServerMCPClient(
    {
        "Spending Habits": {
            "url": "http://localhost:8010/mcp/",
            "transport": "streamable-http",
        },
        "Risk Estimator": {
            "url": "http://localhost:8020/mcp/",
            "transport": "streamable-http",
        },
        "Financial Trends": {
            "url": "http://localhost:8000/mcp/",
            "transport": "streamable-http",
        }
    }
)

In [4]:
tools_mcp = await mcp_client.get_tools()

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain_tavily import TavilySearch

tool_tavily=TavilySearch(max_results=2)

In [7]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [8]:
tools_all = [*tools_mcp, tool_tavily]

In [9]:
import httpx
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    http_client=httpx.Client(verify=False),
    http_async_client=httpx.AsyncClient(verify=False)
)

In [10]:
llm_with_tools = llm.bind_tools(tools_all)

In [11]:
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

In [12]:
async def tool_calling_llm(state: State):
    response = await llm_with_tools.ainvoke(state["messages"])  # ✅ FULL list
    state["messages"].append(response)
    return state

In [13]:
from langchain_core.messages import AIMessage

def check_finish(state: State):
    state["messages"].append(AIMessage(content="Do you have any other queries? (Type 'exit' to end)"))
    return state

In [14]:
def check_finish_router(state: State):
    last_message = state["messages"][-1]
    content = last_message.content if hasattr(last_message, "content") else last_message["content"]

    if content.strip().lower() in ["exit", "quit", "no"]:
        return "end"

    return "continue"

In [15]:

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools_all))
builder.add_node("check_finish", check_finish)

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools", "tool_calling_llm")
builder.add_conditional_edges(
    "check_finish",
    check_finish_router,
    {"end": END, "continue": "tool_calling_llm"}
)

In [16]:
config={"configurable":{"thread_id":"1"}}

In [17]:
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

graph=builder.compile(checkpointer=memory)

state = {
    "messages": [
        SystemMessage(
            content="""
You are a financial assistant.

You MUST call the appropriate tool when all required inputs are present.

Do NOT:
- ask follow-up questions
- validate percentages
- respond in natural language

ONLY call the tool with correct arguments.
"""
        )
    ]
}

while True:
    user_input = input("User: ").strip()
    if user_input.lower() in ["exit", "quit", "no"]:
        print("Assistant: Goodbye!")
        break

    state["messages"].append({
        "role": "user",
        "content": user_input
    })

    state = await graph.ainvoke(state, config=config)

    print("User:", user_input)

    print("Assistant:", state["messages"][-1].content)

User: Hi, I am Abhijit
Assistant: Hello Abhijit! How can I assist you today?
User: I need an analysis of my spending habit
Assistant: Please provide the following details for the analysis:

1. Age
2. Number of Dependents
3. City Tier (1 for metropolitan areas, 2 for smaller cities, 3 for rural areas)
4. Rent as a Percentage of Income
5. Loan Repayment as a Percentage of Income
6. Insurance as a Percentage of Income
7. Groceries as a Percentage of Income
8. Transport as a Percentage of Income
9. Eating Out as a Percentage of Income
10. Entertainment as a Percentage of Income
11. Utilities as a Percentage of Income
12. Healthcare as a Percentage of Income
13. Education as a Percentage of Income
14. Miscellaneous as a Percentage of Income
15. Disposable Income as a Percentage of Income

Once you provide these details, I can proceed with the analysis.
User: {   "Age": 34,   "Dependents": 2,   "City_Tier": 1,   "Rent_As_Percentage_Of_Income": 18.0,   "Loan_Repayment_As_Percentage_Of_Income"